In [54]:
import polars as pl
import json
from collections import defaultdict
from scipy.spatial.distance import jensenshannon
import numpy as np
import itertools

In [55]:
df = pl.read_csv("../dataset/Capstone2025_nsi_lvl9_with_landcover_and_color.csv.gz")

with open("../dataset/ca-county-neighbors.json") as f:
    neighbors_raw = json.load(f)

neighbors = pl.DataFrame(neighbors_raw).with_columns(
    pl.col("county_fips").cast(pl.String).str.zfill(5),
    pl.col("neighbor_fips").cast(pl.String).str.zfill(5),
)

df= df.with_columns(pl.col("fips").cast(pl.String).str.zfill(5))


In [56]:
df

h3,fips,st_damcat,bldgtype,lc_type,loc,clr,clr_cc
str,str,str,str,str,str,str,i64
"""8929a960d6fffff""","""06047""","""RES""","""M""","""urban+crop""","""POINT(-120.828147398273 37.072…","""orange""",8
"""8929a978c07ffff""","""06047""","""RES""","""M""","""urban+crop""","""POINT(-120.63428156983 36.9838…","""orange""",8
"""892836da147ffff""","""06047""","""RES""","""M""","""urban+crop""","""POINT(-120.464731172229 37.323…","""orange""",11
"""892836c644fffff""","""06047""","""RES""","""M""","""urban+crop""","""POINT(-120.787252301526 37.429…","""orange""",12
"""892836d736fffff""","""06047""","""RES""","""M""","""urban+crop""","""POINT(-120.716239893409 37.339…","""orange""",1
…,…,…,…,…,…,…,…
"""892836da15bffff""","""06047""","""RES""","""M""","""urban+crop""","""POINT(-120.457547536701 37.320…","""olive""",6
"""892836c226bffff""","""06047""","""RES""","""M""","""urban+crop""","""POINT(-120.591661571591 37.401…","""olive""",1
"""892836d121bffff""","""06047""","""RES""","""M""","""urban+crop""","""POINT(-120.492292055735 37.305…","""olive""",1


In [57]:
group_cols = ["fips", "st_damcat", "bldgtype", "lc_type"]

color_counts = (
    df.group_by(group_cols + ["clr"])
    .agg(pl.col("clr_cc").sum().alias("count"))
)
color_counts


fips,st_damcat,bldgtype,lc_type,clr,count
str,str,str,str,str,i64
"""06001""","""COM""","""M""","""urban+forest""","""purple""",1
"""06039""","""RES""","""W""","""crop""","""scarlet""",3
"""06045""","""COM""","""C""","""urban+forest""","""coffee""",130
"""06015""","""COM""","""M""","""urban+forest""","""bar""",5
"""06067""","""RES""","""C""","""urban+grass""","""navy""",4
…,…,…,…,…,…
"""06071""","""COM""","""M""","""urban""","""red""",247
"""06087""","""RES""","""H""","""forest""","""terracotta""",31
"""06039""","""RES""","""M""","""urban+grass""","""scarlet""",75


In [58]:
group_totals = (
    color_counts.group_by(group_cols)
    .agg(pl.col("count").sum().alias("total"))
)
group_totals

fips,st_damcat,bldgtype,lc_type,total
str,str,str,str,i64
"""06053""","""RES""","""C""","""urban+forest""",85
"""06109""","""RES""","""H""","""urban+shrub""",2
"""06061""","""IND""","""M""","""urban""",19
"""06039""","""IND""","""H""","""forest""",15
"""06053""","""RES""","""W""","""urban+shrub""",857
…,…,…,…,…
"""06051""","""PUB""","""C""","""urban+grass""",2
"""06031""","""PUB""","""M""","""urban+crop""",2
"""06085""","""RES""","""M""","""crop""",194


In [59]:
color_freq = (
    color_counts.join(group_totals, on=group_cols)
    .with_columns(
        (pl.col("count") / pl.col("total")).alias("freq"),
    )
)

color_freq

fips,st_damcat,bldgtype,lc_type,clr,count,total,freq
str,str,str,str,str,i64,i64,f64
"""06001""","""COM""","""M""","""urban+forest""","""purple""",1,69,0.014493
"""06039""","""RES""","""W""","""crop""","""scarlet""",3,225,0.013333
"""06045""","""COM""","""C""","""urban+forest""","""coffee""",130,325,0.4
"""06015""","""COM""","""M""","""urban+forest""","""bar""",5,67,0.074627
"""06067""","""RES""","""C""","""urban+grass""","""navy""",4,41,0.097561
…,…,…,…,…,…,…,…
"""06071""","""COM""","""M""","""urban""","""red""",247,2476,0.099758
"""06087""","""RES""","""H""","""forest""","""terracotta""",31,292,0.106164
"""06039""","""RES""","""M""","""urban+grass""","""scarlet""",75,1099,0.068244


In [60]:
strata_cols = ["st_damcat", "bldgtype", "lc_type"]
TARGET_MIN_ROWS = 100
AVG_BLDG_PER_ROW = df["clr_cc"].sum() / df.height
MIN_TOTAL = int(round(TARGET_MIN_ROWS * AVG_BLDG_PER_ROW))
MIN_ROWS = TARGET_MIN_ROWS

MIN_VOTES = 30
MAX_ROUNDS = 50
STOP_RATIO = 0.12
TOP_K_EXCLUSIVE = 2
RANK_DECAY = 0.5
MAX_MERGES_PER_ROUND = 6

color_totals = df.group_by("clr").agg(pl.col("clr_cc").sum().alias("total"))
color_total_map = dict(zip(color_totals["clr"].to_list(), color_totals["total"].to_list()))

current_df = df.clone()
master_map = {}
round1_votes = None

def get_canonical(clr):
    while clr in master_map and master_map[clr] != clr:
        clr = master_map[clr]
    return clr

nb_pairs = []
seen = set()
for row in neighbors.iter_rows(named=True):
    c1, c2 = row["county_fips"], row["neighbor_fips"]
    key = (min(c1, c2), max(c1, c2))
    if key not in seen:
        seen.add(key)
        nb_pairs.append(key)

for round_num in range(1, MAX_ROUNDS + 1):
    cc = current_df.group_by(["fips"] + strata_cols + ["clr"]).agg(pl.col("clr_cc").sum().alias("count"), pl.len().alias("n_rows"))
    gt = cc.group_by(["fips"] + strata_cols).agg(pl.col("count").sum().alias("total"), pl.col("n_rows").sum().alias("total_rows"))
    cf = (
        cc.join(gt.filter((pl.col("total") >= MIN_TOTAL) & (pl.col("total_rows") >= MIN_ROWS)), on=["fips"] + strata_cols)
        .with_columns((pl.col("count") / pl.col("total")).alias("freq"))
    )

    pair_votes = defaultdict(float)
    for c1, c2 in nb_pairs:
        c1_strata = cf.filter(pl.col("fips") == c1).select(*strata_cols).unique()

        for s_row in c1_strata.iter_rows(named=True):
            c1_data = (
                cf.filter(
                    (pl.col("fips") == c1) &
                    (pl.col("st_damcat") == s_row["st_damcat"]) &
                    (pl.col("bldgtype") == s_row["bldgtype"]) &
                    (pl.col("lc_type") == s_row["lc_type"])
                ).sort("freq", descending=True)
            )
            c2_data = (
                cf.filter(
                    (pl.col("fips") == c2) &
                    (pl.col("st_damcat") == s_row["st_damcat"]) &
                    (pl.col("bldgtype") == s_row["bldgtype"]) &
                    (pl.col("lc_type") == s_row["lc_type"])
                ).sort("freq", descending=True)
            )
            if c1_data.height == 0 or c2_data.height == 0:
                continue

            c1_top_freq = c1_data["freq"][0]
            c2_top_freq = c2_data["freq"][0]
            c1_avg = 1.0 / c1_data.height
            c2_avg = 1.0 / c2_data.height
            if c1_top_freq < 1.5 * c1_avg or c2_top_freq < 1.5 * c2_avg:
                continue

            c1_ranked = c1_data["clr"].to_list()
            c2_ranked = c2_data["clr"].to_list()

            shared = set(c1_ranked) & set(c2_ranked)
            c1_excl = [c for c in c1_ranked if c not in shared]
            c2_excl = [c for c in c2_ranked if c not in shared]

            k = min(TOP_K_EXCLUSIVE, len(c1_excl), len(c2_excl))
            for i in range(k):
                a, b = min(c1_excl[i], c2_excl[i]), max(c1_excl[i], c2_excl[i])
                pair_votes[(a, b)] += (RANK_DECAY ** i)

    candidates = sorted(
        [(pair, v) for pair, v in pair_votes.items() if v >= MIN_VOTES],
        key=lambda x: x[1],
        reverse=True,
    )
    if not candidates:
        print(f"Round {round_num}: no pairs with >= {MIN_VOTES:g} weighted votes")
        break

    top_votes = candidates[0][1]

    if round_num == 1:
        round1_votes = top_votes
        print(f"Round 1 baseline: {round1_votes:.1f} votes, will stop below {round1_votes * STOP_RATIO:.1f}")

    if top_votes < round1_votes * STOP_RATIO:
        print(
            f"Round {round_num}: top votes ({top_votes:.1f}) < {STOP_RATIO:.0%} of round 1 ({round1_votes:.1f}) — stopping"
        )
        break

    claimed = set()
    round_merges = []
    for (a, b), votes in candidates:
        if len(round_merges) >= MAX_MERGES_PER_ROUND:
            break
        if a in claimed or b in claimed:
            continue

        freq_a = color_total_map.get(a, 0)
        freq_b = color_total_map.get(b, 0)
        canonical, merged = (a, b) if freq_a >= freq_b else (b, a)

        round_merges.append((merged, canonical, votes))
        claimed.add(a)
        claimed.add(b)

    if not round_merges:
        print(f"Round {round_num}: no non-overlapping pairs to merge")
        break

    print(
        f"Round {round_num}: {len(round_merges)} merges (top vote {top_votes:.1f}, {top_votes / round1_votes:.0%} of baseline)"
    )
    for merged, canonical, votes in round_merges:
        print(f"  {merged} -> {canonical} ({votes:.1f} weighted votes)")

    round_map = {merged: canonical for merged, canonical, _ in round_merges}
    for merged, canonical in round_map.items():
        master_map[merged] = canonical
        for k, v in master_map.items():
            if v == merged:
                master_map[k] = canonical

    current_df = current_df.with_columns(pl.col("clr").replace(round_map))

groups = {}
for orig in master_map:
    canon = get_canonical(orig)
    groups.setdefault(canon, set()).add(orig)
for canon in list(groups.keys()):
    groups[canon].add(canon)

final_map = {}
print(f"--- Final groups ---")
for canon in sorted(groups, key=lambda c: -len(groups[c])):
    members = sorted(groups[canon])
    best = max(members, key=lambda c: color_total_map.get(c, 0))
    for m in members:
        final_map[m] = best
    print(f"  {best} <- {members}")

merged_colors = set(final_map.keys())
all_colors_list = sorted(df["clr"].unique().to_list())
singletons = [c for c in all_colors_list if c not in merged_colors]
print(f"Singletons ({len(singletons)}): {singletons}")


Round 1 baseline: 371.5 votes, will stop below 44.6
Round 1: 4 merges (top vote 371.5, 100% of baseline)
  brown -> cocoa (371.5 weighted votes)
  terracotta -> orange (92.5 weighted votes)
  sage -> green (48.5 weighted votes)
  blue -> azure (46.0 weighted votes)
Round 2: 4 merges (top vote 212.0, 57% of baseline)
  coffee -> cocoa (212.0 weighted votes)
  green -> olive (93.5 weighted votes)
  lavender -> navy (82.5 weighted votes)
  indigo -> azure (53.5 weighted votes)
Round 3: 5 merges (top vote 128.0, 34% of baseline)
  azure -> purple (128.0 weighted votes)
  beige -> cocoa (93.0 weighted votes)
  foo -> red (59.5 weighted votes)
  verde -> olive (54.5 weighted votes)
  gray -> alabaster (52.0 weighted votes)
Round 4: 4 merges (top vote 69.0, 19% of baseline)
  sienna -> orange (69.0 weighted votes)
  grey -> alabaster (68.0 weighted votes)
  lilac -> navy (51.0 weighted votes)
  purple -> red (30.5 weighted votes)
Round 5: 4 merges (top vote 79.0, 21% of baseline)
  ivory -> a

In [61]:
LAPLACE_PSEUDOCOUNT = 1
MIN_SUPPORT = 30

neighbors_pairs = (
    neighbors
    .rename({"county_fips": "fips_a", "neighbor_fips": "fips_b"})
    .filter(pl.col("fips_a") < pl.col("fips_b"))
)
adjacency_list = [(row["fips_a"], row["fips_b"]) for row in neighbors_pairs.iter_rows(named=True)]

all_colors_raw = df["clr"].unique().sort().to_list()
all_colors = sorted({final_map.get(c, c) for c in all_colors_raw})
all_lc_types = df["lc_type"].unique().sort().to_list()

county_lc_clr_counts = df.group_by(["fips", "lc_type", "clr"]).agg(pl.col("clr_cc").sum().alias("count"))
county_lc_support = county_lc_clr_counts.group_by(["fips", "lc_type"]).agg(pl.col("count").sum().alias("support"))
support_dict = {(row["fips"], row["lc_type"]): row["support"] for row in county_lc_support.iter_rows(named=True)}


def get_merged_color_distribution(fips_val, lc_type_val):
    subset = county_lc_clr_counts.filter((pl.col("fips") == fips_val) & (pl.col("lc_type") == lc_type_val))
    raw_counts = dict(zip(subset["clr"].to_list(), subset["count"].to_list()))

    merged_counts = {}
    for color, count in raw_counts.items():
        key = final_map.get(color, color)
        merged_counts[key] = merged_counts.get(key, 0) + count

    smoothed = np.array([merged_counts.get(c, 0) + LAPLACE_PSEUDOCOUNT for c in all_colors], dtype=float)
    return smoothed / smoothed.sum()


results = []
for fips_a, fips_b in adjacency_list:
    pair_jsds = []
    pair_supports = []

    for lc in all_lc_types:
        support_a = support_dict.get((fips_a, lc), 0)
        support_b = support_dict.get((fips_b, lc), 0)
        if support_a < MIN_SUPPORT or support_b < MIN_SUPPORT:
            continue

        dist_a = get_merged_color_distribution(fips_a, lc)
        dist_b = get_merged_color_distribution(fips_b, lc)
        jsd = float(jensenshannon(dist_a, dist_b))
        pair_jsds.append(jsd)
        pair_supports.append(min(support_a, support_b))

    if pair_jsds:
        weighted_jsd = float(sum(j * s for j, s in zip(pair_jsds, pair_supports)) / sum(pair_supports))
        results.append({
            "fips_a": fips_a,
            "fips_b": fips_b,
            "weighted_jsd": weighted_jsd,
            "mean_jsd": float(sum(pair_jsds) / len(pair_jsds)),
            "n_shared_lc": len(pair_jsds),
            "total_support": int(sum(pair_supports)),
        })

jsd_results_df = pl.DataFrame(results).sort("weighted_jsd", descending=True)

jsd_stats_greedy_9 = {
    "total_pairs": len(results),
    "n_raw_colors": len(all_colors_raw),
    "n_merged_colors": len(all_colors),
    "mean_jsd": float(np.mean([r["weighted_jsd"] for r in results])) if results else 0.0,
    "max_jsd": float(np.max([r["weighted_jsd"] for r in results])) if results else 0.0,
    "min_jsd": float(np.min([r["weighted_jsd"] for r in results])) if results else 0.0,
}

jsd_stats_greedy_9

{'total_pairs': 144,
 'n_raw_colors': 38,
 'n_merged_colors': 13,
 'mean_jsd': 0.2162653400182195,
 'max_jsd': 0.6055784281783269,
 'min_jsd': 0.03824187517097325}

In [ ]:
ALPHA = 1.0 
BIC_PENALTY = 1.0 
MIN_GROUP_SIZE = 5
MIN_SPLIT_GAIN = 1_500_000.0
MAX_SPLIT_ROUNDS = 20
MAX_COLORS_FOR_EXACT = 12

ctx_key_cols = ["fips", "st_damcat", "bldgtype", "lc_type"]
ctx_counts = (
    df.group_by(ctx_key_cols + ["clr"])
    .agg(pl.col("clr_cc").sum().alias("count"))
    .with_columns(pl.concat_str(ctx_key_cols, separator="|").alias("ctx"))
    .select("clr", "ctx", "count")
)

all_colors = sorted(df["clr"].unique().to_list())
all_contexts = sorted(ctx_counts["ctx"].unique().to_list())

color_to_idx = {c: i for i, c in enumerate(all_colors)}
ctx_to_idx = {c: i for i, c in enumerate(all_contexts)}

X = np.zeros((len(all_colors), len(all_contexts)), dtype=float)
for row in ctx_counts.iter_rows(named=True):
    X[color_to_idx[row["clr"]], ctx_to_idx[row["ctx"]]] = float(row["count"])

color_total_map_refine = {c: float(X[color_to_idx[c]].sum()) for c in all_colors}


def build_groups_from_map(color_map, colors):
    tmp = defaultdict(list)
    for c in colors:
        tmp[color_map.get(c, c)].append(c)

    out = {}
    for _, members in tmp.items():
        canon = max(members, key=lambda x: color_total_map_refine.get(x, 0.0))
        out[canon] = sorted(set(members))
    return out


def cluster_penalized_score(member_idxs, alpha=ALPHA, bic_penalty=BIC_PENALTY):
    if not member_idxs:
        return -np.inf, -np.inf, np.inf, 0.0

    sub = X[member_idxs, :]
    pooled = sub.sum(axis=0)
    total = float(pooled.sum())
    if total <= 0:
        return -np.inf, -np.inf, np.inf, total

    active = pooled > 0
    k_eff = int(active.sum())
    if k_eff == 0:
        return -np.inf, -np.inf, np.inf, total

    sub_active = sub[:, active]
    pooled_active = pooled[active]

    theta = (pooled_active + alpha) / (total + alpha * k_eff)
    loglik = float((sub_active * np.log(theta)).sum())

    n_params = max(k_eff - 1, 1)
    penalty = 0.5 * bic_penalty * n_params * np.log(total + 1.0)
    score = loglik - penalty
    return score, loglik, penalty, total


def group_partition_score(groups_dict):
    total_score = 0.0
    for members in groups_dict.values():
        idxs = [color_to_idx[c] for c in members]
        total_score += cluster_penalized_score(idxs)[0]
    return float(total_score)


def best_binary_split_exact(members):
    n = len(members)
    if n < 2 or n > MAX_COLORS_FOR_EXACT:
        return None

    full_idxs = [color_to_idx[c] for c in members]
    base_score, base_ll, base_pen, _ = cluster_penalized_score(full_idxs)

    anchor = members[0]
    best = None

    for r in range(1, n):
        for subset in itertools.combinations(members[1:], r - 1):
            left = [anchor, *subset]
            if len(left) == n:
                continue
            left_set = set(left)
            right = [c for c in members if c not in left_set]
            if not right:
                continue

            left_idxs = [color_to_idx[c] for c in left]
            right_idxs = [color_to_idx[c] for c in right]

            s_left, ll_left, pen_left, _ = cluster_penalized_score(left_idxs)
            s_right, ll_right, pen_right, _ = cluster_penalized_score(right_idxs)
            split_score = s_left + s_right
            gain = split_score - base_score

            if (best is None) or (split_score > best["split_score"]):
                best = {
                    "left": sorted(left),
                    "right": sorted(right),
                    "base_score": base_score,
                    "split_score": split_score,
                    "gain": gain,
                    "base_loglik": base_ll,
                    "split_loglik": ll_left + ll_right,
                    "base_penalty": base_pen,
                    "split_penalty": pen_left + pen_right,
                }

    return best


orig_groups = build_groups_from_map(final_map, all_colors)
work_groups = {k: v[:] for k, v in orig_groups.items()}

split_events = []
for round_id in range(1, MAX_SPLIT_ROUNDS + 1):
    changed = False
    next_groups = {}

    for _, members in sorted(work_groups.items(), key=lambda kv: (-len(kv[1]), kv[0])):
        members = sorted(members)
        if len(members) < MIN_GROUP_SIZE:
            canon = max(members, key=lambda x: color_total_map_refine.get(x, 0.0))
            next_groups[canon] = members
            continue

        candidate = best_binary_split_exact(members)
        if candidate is None or candidate["gain"] <= MIN_SPLIT_GAIN:
            canon = max(members, key=lambda x: color_total_map_refine.get(x, 0.0))
            next_groups[canon] = members
            continue

        left = candidate["left"]
        right = candidate["right"]
        left_canon = max(left, key=lambda x: color_total_map_refine.get(x, 0.0))
        right_canon = max(right, key=lambda x: color_total_map_refine.get(x, 0.0))

        next_groups[left_canon] = left
        next_groups[right_canon] = right
        changed = True

        split_events.append(
            {
                "round": round_id,
                "before": members,
                "after_left": left,
                "after_right": right,
                "gain": float(candidate["gain"]),
                "base_score": float(candidate["base_score"]),
                "split_score": float(candidate["split_score"]),
                "base_loglik": float(candidate["base_loglik"]),
                "split_loglik": float(candidate["split_loglik"]),
                "base_penalty": float(candidate["base_penalty"]),
                "split_penalty": float(candidate["split_penalty"]),
            }
        )

    work_groups = next_groups
    if not changed:
        print(f"Split refinement converged at round {round_id}: no accepted splits.")
        break

refined_groups = work_groups
refined_final_map = {}
for canon, members in refined_groups.items():
    for c in members:
        refined_final_map[c] = canon

orig_score = group_partition_score(orig_groups)
refined_score = group_partition_score(refined_groups)

print("--- Split refinement summary ---")
print(f"Original groups: {len(orig_groups)}")
print(f"Refined groups:  {len(refined_groups)}")
print(f"Penalized score (orig):    {orig_score:,.2f}")
print(f"Penalized score (refined): {refined_score:,.2f}")
print(f"Score gain:                {refined_score - orig_score:,.2f}")

if split_events:
    print()
    print("Accepted splits:")
    for ev in split_events:
        print(
            f"  round {ev['round']}: {ev['before']} -> {ev['after_left']} + {ev['after_right']} "
            f"(gain={ev['gain']:.2f})"
        )
else:
    print()
    print("No splits passed the penalized-likelihood threshold.")

print()
print("--- Refined groups ---")
for canon in sorted(refined_groups, key=lambda c: (-len(refined_groups[c]), c)):
    print(f"  {canon} <- {refined_groups[canon]}")

refined_singletons = sorted([canon for canon, m in refined_groups.items() if len(m) == 1])
print(f"Refined singletons ({len(refined_singletons)}): {refined_singletons}")


Split refinement converged at round 1: no accepted splits.
--- Split refinement summary ---
Original groups: 13
Refined groups:  13
Penalized score (orig):    -23,611,621.92
Penalized score (refined): -23,611,621.92
Score gain:                -0.00

No splits passed the penalized-likelihood threshold.

--- Refined groups ---
  red <- ['azure', 'blue', 'crimson', 'foo', 'indigo', 'purple', 'red', 'scarlet']
  navy <- ['aqua', 'aquamarine', 'lavender', 'lilac', 'navy']
  alabaster <- ['alabaster', 'gray', 'grey', 'ivory']
  amber <- ['amber', 'gold', 'lemon', 'yellow']
  cocoa <- ['beige', 'brown', 'cocoa', 'coffee']
  olive <- ['green', 'olive', 'sage', 'verde']
  orange <- ['orange', 'sienna', 'terracotta']
  auburn <- ['auburn']
  bar <- ['bar']
  emerald <- ['emerald']
  maroon <- ['maroon']
  plum <- ['plum']
  tan <- ['tan']
Refined singletons (6): ['auburn', 'bar', 'emerald', 'maroon', 'plum', 'tan']


In [66]:
LAPLACE_PSEUDOCOUNT = 1
MIN_SUPPORT = 30

neighbors_pairs = (
    neighbors
    .rename({"county_fips": "fips_a", "neighbor_fips": "fips_b"})
    .filter(pl.col("fips_a") < pl.col("fips_b"))
)
adjacency_list = [(row["fips_a"], row["fips_b"]) for row in neighbors_pairs.iter_rows(named=True)]

all_colors_raw = df["clr"].unique().sort().to_list()
all_colors_refined = sorted({refined_final_map.get(c, c) for c in all_colors_raw})
all_lc_types = df["lc_type"].unique().sort().to_list()

county_lc_clr_counts = df.group_by(["fips", "lc_type", "clr"]).agg(pl.col("clr_cc").sum().alias("count"))
county_lc_support = county_lc_clr_counts.group_by(["fips", "lc_type"]).agg(pl.col("count").sum().alias("support"))
support_dict = {(row["fips"], row["lc_type"]): row["support"] for row in county_lc_support.iter_rows(named=True)}


def get_refined_distribution(fips_val, lc_type_val):
    subset = county_lc_clr_counts.filter((pl.col("fips") == fips_val) & (pl.col("lc_type") == lc_type_val))
    raw_counts = dict(zip(subset["clr"].to_list(), subset["count"].to_list()))

    merged_counts = {}
    for color, count in raw_counts.items():
        key = refined_final_map.get(color, color)
        merged_counts[key] = merged_counts.get(key, 0) + count

    smoothed = np.array([merged_counts.get(c, 0) + LAPLACE_PSEUDOCOUNT for c in all_colors_refined], dtype=float)
    return smoothed / smoothed.sum()


refined_results = []
for fips_a, fips_b in adjacency_list:
    pair_jsds = []
    pair_supports = []

    for lc in all_lc_types:
        support_a = support_dict.get((fips_a, lc), 0)
        support_b = support_dict.get((fips_b, lc), 0)
        if support_a < MIN_SUPPORT or support_b < MIN_SUPPORT:
            continue

        dist_a = get_refined_distribution(fips_a, lc)
        dist_b = get_refined_distribution(fips_b, lc)
        jsd = float(jensenshannon(dist_a, dist_b))
        pair_jsds.append(jsd)
        pair_supports.append(min(support_a, support_b))

    if pair_jsds:
        weighted_jsd = float(sum(j * s for j, s in zip(pair_jsds, pair_supports)) / sum(pair_supports))
        refined_results.append(
            {
                "fips_a": fips_a,
                "fips_b": fips_b,
                "weighted_jsd": weighted_jsd,
                "mean_jsd": float(sum(pair_jsds) / len(pair_jsds)),
                "n_shared_lc": len(pair_jsds),
                "total_support": int(sum(pair_supports)),
            }
        )

jsd_results_refined = pl.DataFrame(refined_results).sort("weighted_jsd", descending=True)

jsd_stats_refined = {
    "total_pairs": len(refined_results),
    "n_raw_colors": len(all_colors_raw),
    "n_refined_colors": len(all_colors_refined),
    "mean_jsd": float(np.mean([r["weighted_jsd"] for r in refined_results])) if refined_results else 0.0,
    "max_jsd": float(np.max([r["weighted_jsd"] for r in refined_results])) if refined_results else 0.0,
    "min_jsd": float(np.min([r["weighted_jsd"] for r in refined_results])) if refined_results else 0.0,
}

print(jsd_stats_refined)
jsd_results_refined.head(20)


{'total_pairs': 144, 'n_raw_colors': 38, 'n_refined_colors': 13, 'mean_jsd': 0.2162653400182195, 'max_jsd': 0.6055784281783269, 'min_jsd': 0.03824187517097325}


fips_a,fips_b,weighted_jsd,mean_jsd,n_shared_lc,total_support
str,str,f64,f64,i64,i64
"""06043""","""06109""",0.605578,0.536606,4,2857
"""06043""","""06099""",0.602753,0.551508,4,2857
"""06003""","""06017""",0.586379,0.586379,1,422
"""06003""","""06005""",0.583043,0.583043,1,422
"""06063""","""06091""",0.580322,0.573592,3,838
…,…,…,…,…,…
"""06103""","""06105""",0.29002,0.294059,2,3215
"""06089""","""06093""",0.28976,0.246537,7,12994
"""06093""","""06105""",0.289175,0.297547,2,3215


# Manual

In [64]:
LAPLACE_PSEUDOCOUNT = 1
MIN_SUPPORT = 30
semantic_groups = {
    "blue": ["aqua", "aquamarine", "azure", "blue", "navy",  "bar"],
    "purple": ["indigo", "lavender", "lilac", "plum", "purple"],
    "red": ["auburn", "crimson", "maroon", "red", "scarlet", "foo"],
    "orange": ["orange", "sienna", "terracotta"],
    "yellow": ["amber", "gold", "lemon", "yellow"],
    "green": ["emerald", "green", "olive", "sage", "verde", ],
    "brown": ["beige", "brown", "cocoa", "coffee","tan"],
    "gray": ["alabaster",  "gray", "grey", "ivory",],
}

semantic_map = {}
for canon, members in semantic_groups.items():
    for c in members:
        semantic_map[c] = canon

all_colors_raw = df["clr"].unique().sort().to_list()
unmapped_colors = sorted([c for c in all_colors_raw if c not in semantic_map])
if unmapped_colors:
    print(f"Unmapped colors (kept as singletons): {unmapped_colors}")

all_colors_semantic = sorted({semantic_map.get(c, c) for c in all_colors_raw})

neighbors_pairs = (
    neighbors
    .rename({"county_fips": "fips_a", "neighbor_fips": "fips_b"})
    .filter(pl.col("fips_a") < pl.col("fips_b"))
)
adjacency_list = [(row["fips_a"], row["fips_b"]) for row in neighbors_pairs.iter_rows(named=True)]
all_lc_types = df["lc_type"].unique().sort().to_list()

county_lc_clr_counts = df.group_by(["fips", "lc_type", "clr"]).agg(pl.col("clr_cc").sum().alias("count"))
county_lc_support = county_lc_clr_counts.group_by(["fips", "lc_type"]).agg(pl.col("count").sum().alias("support"))
support_dict = {(row["fips"], row["lc_type"]): row["support"] for row in county_lc_support.iter_rows(named=True)}


def get_semantic_distribution(fips_val, lc_type_val):
    subset = county_lc_clr_counts.filter((pl.col("fips") == fips_val) & (pl.col("lc_type") == lc_type_val))
    raw_counts = dict(zip(subset["clr"].to_list(), subset["count"].to_list()))

    merged_counts = {}
    for color, count in raw_counts.items():
        key = semantic_map.get(color, color)
        merged_counts[key] = merged_counts.get(key, 0) + count

    smoothed = np.array([merged_counts.get(c, 0) + LAPLACE_PSEUDOCOUNT for c in all_colors_semantic], dtype=float)
    return smoothed / smoothed.sum()


semantic_results = []
for fips_a, fips_b in adjacency_list:
    pair_jsds = []
    pair_supports = []

    for lc in all_lc_types:
        support_a = support_dict.get((fips_a, lc), 0)
        support_b = support_dict.get((fips_b, lc), 0)
        if support_a < MIN_SUPPORT or support_b < MIN_SUPPORT:
            continue

        dist_a = get_semantic_distribution(fips_a, lc)
        dist_b = get_semantic_distribution(fips_b, lc)
        jsd = float(jensenshannon(dist_a, dist_b))
        pair_jsds.append(jsd)
        pair_supports.append(min(support_a, support_b))

    if pair_jsds:
        weighted_jsd = float(sum(j * s for j, s in zip(pair_jsds, pair_supports)) / sum(pair_supports))
        semantic_results.append(
            {
                "fips_a": fips_a,
                "fips_b": fips_b,
                "weighted_jsd": weighted_jsd,
                "mean_jsd": float(sum(pair_jsds) / len(pair_jsds)),
                "n_shared_lc": len(pair_jsds),
                "total_support": int(sum(pair_supports)),
            }
        )

jsd_results_semantic = pl.DataFrame(semantic_results).sort("weighted_jsd", descending=True)

mean_weighted_jsd_semantic = float(np.mean([r["weighted_jsd"] for r in semantic_results])) if semantic_results else 0.0
mean_unweighted_jsd_semantic = float(np.mean([r["mean_jsd"] for r in semantic_results])) if semantic_results else 0.0

print({
    "n_pairs": len(semantic_results),
    "n_semantic_colors": len(all_colors_semantic),
    "mean_weighted_jsd": mean_weighted_jsd_semantic,
    "mean_unweighted_jsd": mean_unweighted_jsd_semantic,
})

jsd_results_semantic.head(20)


{'n_pairs': 144, 'n_semantic_colors': 8, 'mean_weighted_jsd': 0.184260905033097, 'mean_unweighted_jsd': 0.18978422675258047}


fips_a,fips_b,weighted_jsd,mean_jsd,n_shared_lc,total_support
str,str,f64,f64,i64,i64
"""06089""","""06093""",0.283409,0.246491,7,12994
"""06063""","""06091""",0.273453,0.230577,3,838
"""06081""","""06085""",0.271245,0.237,6,109707
"""06095""","""06113""",0.270756,0.268428,7,11609
"""06089""","""06103""",0.269562,0.248822,7,10257
…,…,…,…,…,…
"""06089""","""06105""",0.249936,0.240983,2,3215
"""06069""","""06087""",0.244705,0.251705,3,662
"""06019""","""06047""",0.243826,0.229317,8,40884
